# Cloud Provider Analytics — Pipeline end-to-end (Fase 2)

**Landing → Bronze → Silver → Gold → Cassandra/AstraDB**

Este notebook implementa el pipeline mínimo de la fase 2. Se ejecuta **de arriba hacia abajo**, sección por sección. Cada sección empieza con una explicación de qué hace, por qué y **qué mirar en la salida**.

| Sección | Qué hace | Tiempo aprox. |
|---|---|---|
| 0 | Configuración, Spark y logging | 3 min |
| 1 | Batch → Bronze: 3 maestros | 2 min |
| 2 | Streaming → Bronze: eventos con watermark, dedup y checkpoint | 5 min |
| 3 | Silver: reglas de calidad, quarantine, joins y features | 3 min |
| 4 | Gold: mart `org_daily_usage_by_service` | 2 min |
| 5 | Analítica: detección de anomalías (MAD) | 2 min |
| 6 | Serving: carga en AstraDB y 2 consultas CQL | 5 min |
| 7 | Idempotencia: re-ejecución sin duplicados | 5 min |
| 8 | Evidencias, tamaños y persistencia en Drive | 2 min |
| 9 | Limpieza y reinicio | — |

**Requisitos previos:** el dataset en Google Drive, una base AstraDB activa con el keyspace `cloud_analytics`, el Secure Connect Bundle en Drive y el token cargado en los *Secretos* de Colab (sección 6).

## 0. Configuración

Toda la configuración vive en una sola celda: rutas, nombres y parámetros. Así el pipeline no tiene valores "escondidos" en el código, y para correrlo en otra cuenta solo hay que cambiar `BASE`.

**Decisión (D-11 actualizada):** el lake se procesa en el disco local de Colab (`/content/lake`), que es rápido, y al final se copia a Drive. Escribir cientos de archivos Parquet y checkpoints directamente sobre Drive es lento y poco confiable.

In [1]:
!pip -q install pyspark cassandra-driver

In [2]:
import os, json, glob, math, shutil, logging, datetime as dt
from google.colab import drive
drive.mount('/content/drive')

# ---------- Configuración (editable) ----------
BASE        = os.environ.get("CPA_BASE", "/content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto")
LANDING     = f"{BASE}/cloud_provider_challenge_dataset_v1/datalake/landing"
EVENTS_PATH = f"{LANDING}/usage_events_stream"
LAKE        = "/content/lake"                      # procesamiento local (rápido)
DRIVE_OUT   = f"{BASE}/datalake_output"            # copia persistente al final
BRONZE, SILVER, GOLD, QUAR = (f"{LAKE}/{z}" for z in ("bronze", "silver", "gold", "quarantine"))
CHK         = f"{LAKE}/_checkpoints"
EVID        = f"{LAKE}/_evidence"

FILES_PER_TRIGGER = 10                             # archivos por micro-lote
ASTRA_KEYSPACE    = "cloud_analytics"
ASTRA_SCB         = f"{BASE}/secrets/secure-connect-cloud-provider-db.zip"
ANOMALY_Z         = 3.5                            # umbral de z-score robusto
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

for p in (BRONZE, SILVER, GOLD, QUAR, CHK, EVID):
    os.makedirs(p, exist_ok=True)
assert os.path.isdir(LANDING), f"No encuentro LANDING. Revisá BASE: {LANDING}"

# ---------- Logging a consola y a archivo ----------
log = logging.getLogger("pipeline")
log.handlers.clear(); log.setLevel(logging.INFO)
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for h in (logging.StreamHandler(), logging.FileHandler(f"{EVID}/run_{RUN_ID}.log")):
    h.setFormatter(fmt); log.addHandler(h)
log.info(f"RUN_ID={RUN_ID} | LANDING={LANDING}")
print("Archivos de eventos:", len(glob.glob(f"{EVENTS_PATH}/*.jsonl")))

Mounted at /content/drive


2026-11-18 19:09:13,707 | INFO | RUN_ID=20261118T190912Z | LANDING=/content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto/cloud_provider_challenge_dataset_v1/datalake/landing
INFO:pipeline:RUN_ID=20261118T190912Z | LANDING=/content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto/cloud_provider_challenge_dataset_v1/datalake/landing


Archivos de eventos: 120


In [3]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               BooleanType, DateType, IntegerType, LongType)

spark = (SparkSession.builder
         .appName("cloud-provider-pipeline")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")  # re-ejecución idempotente
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

def tamano_tabla(path):
    """Evidencia de performance: archivos, MB y particiones de una tabla Parquet."""
    files = [f for f in glob.glob(f"{path}/**/*.parquet", recursive=True)]
    parts = {os.path.dirname(f) for f in files}
    mb = sum(os.path.getsize(f) for f in files) / 1e6
    return {"tabla": path.replace(LAKE + "/", ""), "archivos": len(files),
            "particiones": len(parts), "MB": round(mb, 3)}

RESUMEN = {"run_id": RUN_ID}   # se va llenando y se guarda como evidencia al final

Spark 4.0.5


## 1. Batch → Bronze: tres maestros

Se ingestan `customers_orgs`, `users` y `billing_monthly` con **esquema explícito** (ya no `inferSchema`: la exploración mostró que la inferencia se equivoca). A cada fila se le agregan las columnas técnicas `ingest_ts`, `source_file` y `run_id`, y se deduplica por la clave natural.

**Particionado:**
- `customers_orgs` y `users` por `ingest_date`: cada carga es una *foto* (snapshot) del maestro. Silver toma siempre la más reciente, y el historial de fotos queda disponible para analizar cambios.
- `billing_monthly` por `month`, su granularidad natural.

**Resultado esperado:** la tabla final. `leidas` debe coincidir con la exploración (80, 800 y 240) y `duplicados` debería ser 0.

In [4]:
SCHEMAS = {
    "customers_orgs": StructType([
        StructField("org_id", StringType()), StructField("org_name", StringType()),
        StructField("industry", StringType()), StructField("hq_region", StringType()),
        StructField("plan_tier", StringType()), StructField("is_enterprise", BooleanType()),
        StructField("signup_date", DateType()), StructField("sales_rep", StringType()),
        StructField("lifecycle_stage", StringType()), StructField("marketing_source", StringType()),
        StructField("nps_score", DoubleType())]),
    "users": StructType([
        StructField("user_id", StringType()), StructField("org_id", StringType()),
        StructField("email", StringType()), StructField("role", StringType()),
        StructField("active", BooleanType()), StructField("created_at", DateType()),
        StructField("last_login", DateType())]),
    "billing_monthly": StructType([
        StructField("invoice_id", StringType()), StructField("org_id", StringType()),
        StructField("month", DateType()), StructField("subtotal", DoubleType()),
        StructField("credits", DoubleType()), StructField("taxes", DoubleType()),
        StructField("currency", StringType()), StructField("exchange_rate_to_usd", DoubleType())]),
}
MAESTROS = {"customers_orgs": ("org_id", "ingest_date"),
            "users": ("user_id", "ingest_date"),
            "billing_monthly": ("invoice_id", "month")}

def ingestar_maestro(nombre):
    clave, particion = MAESTROS[nombre]
    df = (spark.read.option("header", True).option("mode", "PERMISSIVE")
          .schema(SCHEMAS[nombre]).csv(f"{LANDING}/{nombre}.csv"))
    n_in = df.count()
    df = (df.withColumn("ingest_ts", F.current_timestamp())
            .withColumn("ingest_date", F.current_date())
            .withColumn("source_file", F.input_file_name())
            .withColumn("run_id", F.lit(RUN_ID))
            .dropDuplicates([clave]))
    n_out = df.count()
    df.write.mode("overwrite").partitionBy(particion).parquet(f"{BRONZE}/{nombre}")
    log.info(f"[batch] {nombre}: leidas={n_in} escritas={n_out}")
    return {"tabla": nombre, "leidas": n_in, "duplicados": n_in - n_out, "escritas": n_out}

import pandas as pd
res_batch = pd.DataFrame([ingestar_maestro(n) for n in MAESTROS])
RESUMEN["batch_bronze"] = res_batch.to_dict("records")
display(res_batch)
spark.read.parquet(f"{BRONZE}/billing_monthly").printSchema()

2026-11-18 19:12:43,869 | INFO | [batch] customers_orgs: leidas=80 escritas=80
INFO:pipeline:[batch] customers_orgs: leidas=80 escritas=80
2026-11-18 19:12:48,024 | INFO | [batch] users: leidas=800 escritas=800
INFO:pipeline:[batch] users: leidas=800 escritas=800
2026-11-18 19:12:50,811 | INFO | [batch] billing_monthly: leidas=240 escritas=240
INFO:pipeline:[batch] billing_monthly: leidas=240 escritas=240


,tabla,leidas,duplicados,escritas
0,customers_orgs,80,0,80
1,users,800,0,800
2,billing_monthly,240,0,240


root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- credits: double (nullable = true)
 |-- taxes: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- month: date (nullable = true)



## 2. Streaming → Bronze: eventos de uso

### 2.1 Diagnóstico: ¿qué tan desordenados llegan los eventos?

El **watermark** le dice a Spark cuánto esperar a los eventos que llegan tarde. Si es muy chico, Spark descarta eventos válidos; si es muy grande, guarda estado de más. En lugar de elegir un número a ojo, lo medimos: Spark lee los archivos en orden de fecha de modificación, en grupos de `FILES_PER_TRIGGER`. Para cada micro-lote calculamos cuánto más viejo es su evento más antiguo respecto del más reciente ya visto. El máximo de ese atraso es el watermark mínimo para no perder datos.

**Importante de esta sesión para hallazgos:** `atraso_max_horas` y el `WATERMARK_HOURS` elegido.

In [5]:
files = sorted(glob.glob(f"{EVENTS_PATH}/*.jsonl"), key=lambda p: (os.path.getmtime(p), p))
mtimes_unicos = len({os.path.getmtime(p) for p in files})

def rango_ts(path):
    ts = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                ts.append(dt.datetime.fromisoformat(json.loads(line)["timestamp"].replace("Z", "+00:00")))
            except Exception:
                pass
    return min(ts), max(ts)

lotes, max_visto, atraso_max = [], None, dt.timedelta(0)
for i in range(0, len(files), FILES_PER_TRIGGER):
    rangos = [rango_ts(p) for p in files[i:i + FILES_PER_TRIGGER]]
    mn, mx = min(r[0] for r in rangos), max(r[1] for r in rangos)
    atraso = (max_visto - mn) if (max_visto and max_visto > mn) else dt.timedelta(0)
    atraso_max = max(atraso_max, atraso)
    max_visto = mx if max_visto is None else max(max_visto, mx)
    lotes.append({"lote": i // FILES_PER_TRIGGER, "min_ts": mn, "max_ts": mx,
                  "atraso_horas": round(atraso.total_seconds() / 3600, 2)})

atraso_max_horas = atraso_max.total_seconds() / 3600
WATERMARK_HOURS = max(1, math.ceil(atraso_max_horas) + 1)   # atraso observado + 1 h de margen
display(pd.DataFrame(lotes))
print(f"Archivos: {len(files)} | fechas de modificación distintas: {mtimes_unicos}")
print(f"Atraso máximo observado: {atraso_max_horas:.2f} h  ->  WATERMARK_HOURS = {WATERMARK_HOURS}")
RESUMEN["watermark_horas"] = WATERMARK_HOURS
RESUMEN["atraso_max_horas"] = round(atraso_max_horas, 2)

,lote,min_ts,max_ts,atraso_horas
0,0,2025-07-03 00:08:00+00:00,2025-08-31 23:56:00+00:00,0.00
1,1,2025-07-03 00:02:00+00:00,2025-08-31 23:51:00+00:00,1439.90
2,2,2025-07-03 00:29:00+00:00,2025-08-31 23:35:00+00:00,1439.45
3,3,2025-07-03 00:35:00+00:00,2025-08-31 23:44:00+00:00,1439.35
4,4,2025-07-03 00:10:00+00:00,2025-08-31 23:46:00+00:00,1439.77
5,5,2025-07-03 00:06:00+00:00,2025-08-31 23:37:00+00:00,1439.83
6,6,2025-07-03 00:05:00+00:00,2025-08-31 23:30:00+00:00,1439.85
7,7,2025-07-03 00:17:00+00:00,2025-08-31 23:44:00+00:00,1439.65
8,8,2025-07-03 00:33:00+00:00,2025-08-31 23:42:00+00:00,1439.38
9,9,2025-07-03 00:17:00+00:00,2025-08-31 23:50:00+00:00,1439.65


Archivos: 120 | fechas de modificación distintas: 3
Atraso máximo observado: 1439.90 h  ->  WATERMARK_HOURS = 1441


### 2.1.b Elección del watermark: evidencia

**Hallazgo.** El diagnóstico de la sección 2.1 muestra que cada micro-lote contiene eventos de los 60 días completos del período (del 03/07/2025 al 31/08/2025). Los archivos de `usage_events_stream` no llegan en orden cronológico: cada uno es una muestra desordenada de todo el histórico. El atraso máximo observado entre micro-lotes es de 1.439,9 horas.

**Consecuencia.** El watermark descarta los eventos cuya fecha es anterior al evento más reciente ya visto, menos la tolerancia configurada. Con este patrón de llegada, un watermark convencional descartaría como "tardíos" eventos perfectamente válidos, sin generar ningún error.

**Simulación.** Se reprodujo la lógica del watermark sobre los 43.200 eventos, procesados en el mismo orden y con los mismos micro-lotes que usa Spark:

| Watermark | Eventos descartados | % perdido |
|---|---|---|
| 1 hora | 39.566 | 91,6 % |
| 1 día | 38.935 | 90,1 % |
| 1 semana | 34.983 | 81,0 % |
| 30 días | 19.758 | 45,7 % |
| **1.441 horas (elegido)** | **0** | **0,0 %** |

**Decisión.** Se adopta un watermark de 1.441 horas: el atraso máximo observado más una hora de margen. Es el valor mínimo que procesa el 100 % de los eventos. El valor no es fijo: se calcula a partir del desorden medido en los datos (`WATERMARK_HOURS`). En un escenario productivo, con eventos que llegan en orden casi real, el mismo cálculo arrojaría una tolerancia de pocas horas, sin cambiar el código.

**Trade-off.** Un watermark amplio obliga a Spark a mantener en estado los `event_id` de toda la ventana para deduplicar. Con el volumen actual (43.200 eventos) el costo es despreciable; en producción, la ventana debería recalcularse periódicamente con el desorden real de llegada.


In [7]:
# Simulación: eventos que se descartarían como tardíos según el watermark elegido
lotes_ts = []
for i in range(0, len(files), FILES_PER_TRIGGER):
    ts = []
    for p in files[i:i + FILES_PER_TRIGGER]:
        with open(p, encoding="utf-8") as f:
            for line in f:
                try:
                    ts.append(dt.datetime.fromisoformat(json.loads(line)["timestamp"].replace("Z", "+00:00")))
                except Exception:
                    pass
    lotes_ts.append(ts)

total = sum(len(t) for t in lotes_ts)
filas = []
for h in [1, 24, 24 * 7, 24 * 30, WATERMARK_HOURS]:
    wm, descartados = None, 0
    for ts in lotes_ts:
        if wm is not None:
            descartados += sum(1 for t in ts if t < wm)
        nuevo = max(ts) - dt.timedelta(hours=h)
        wm = nuevo if wm is None else max(wm, nuevo)
    filas.append({"watermark_horas": h, "eventos_descartados": descartados,
                  "pct_perdido": round(100 * descartados / total, 1)})

sim = pd.DataFrame(filas)
sim.to_csv(f"{EVID}/simulacion_watermark.csv", index=False)
RESUMEN["simulacion_watermark"] = sim.to_dict("records")
display(sim)

,watermark_horas,eventos_descartados,pct_perdido
0,1,39566,91.6
1,24,38935,90.1
2,168,34983,81.0
3,720,19758,45.7
4,1441,0,0.0


### 2.2 Esquema explícito y stream

- `value` y `timestamp` se leen como **texto** y se convierten con `try_cast` / `try_to_timestamp`: si un valor no se puede convertir, queda nulo y la regla de calidad lo envía a quarantine, en lugar de romper el pipeline (D-06).
- `_corrupt_record` captura líneas JSON malformadas.
- `dropDuplicatesWithinWatermark(["event_id"])` deduplica dentro de la ventana del watermark.
- `foreachBatch` escribe cada micro-lote en Bronze y, además, **descarta los `event_id` que ya están en Bronze** (anti-join). Es una segunda capa de idempotencia: aunque se borre el checkpoint y se reprocese todo, no se duplica.
- `trigger(availableNow=True)` procesa todos los archivos disponibles en varios micro-lotes y termina solo. Es ideal en un notebook.

**Resultado esperado:** el log de cada micro-lote y la conciliación final. `landing` y `bronze` deben coincidir (43.200) y `descartados_tardios` debería ser 0.

In [6]:
EVENT_SCHEMA = StructType([
    StructField("event_id", StringType()),     StructField("timestamp", StringType()),
    StructField("org_id", StringType()),       StructField("resource_id", StringType()),
    StructField("service", StringType()),      StructField("region", StringType()),
    StructField("metric", StringType()),       StructField("value", StringType()),
    StructField("unit", StringType()),         StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()),    StructField("genai_tokens", LongType()),
    StructField("_corrupt_record", StringType())])

BRONZE_EVENTS = f"{BRONZE}/usage_events"

def escribir_bronze(batch_df, batch_id):
    batch_df = batch_df.persist()
    if glob.glob(f"{BRONZE_EVENTS}/**/*.parquet", recursive=True):
        existentes = spark.read.parquet(BRONZE_EVENTS).select("event_id")
        batch_df = batch_df.join(existentes, "event_id", "left_anti")   # idempotencia por clave natural
    n = batch_df.count()
    batch_df.write.mode("append").partitionBy("event_date").parquet(BRONZE_EVENTS)
    log.info(f"[stream] micro-lote {batch_id}: {n} filas nuevas en Bronze")
    batch_df.unpersist()

def correr_stream(checkpoint):
    raw = (spark.readStream.schema(EVENT_SCHEMA)
           .option("maxFilesPerTrigger", FILES_PER_TRIGGER)
           .option("columnNameOfCorruptRecord", "_corrupt_record")
           .json(EVENTS_PATH))
    ev = (raw.withColumn("ts", F.expr("try_to_timestamp(timestamp)"))
             .withColumn("value_num", F.expr("try_cast(value AS DOUBLE)"))
             .withColumn("event_date", F.to_date("ts"))
             .withColumn("ingest_ts", F.current_timestamp())
             .withColumn("source_file", F.input_file_name())
             .withColumn("run_id", F.lit(RUN_ID)))
    dedup = (ev.withWatermark("ts", f"{WATERMARK_HOURS} hours")
               .dropDuplicatesWithinWatermark(["event_id"]))
    q = (dedup.writeStream.foreachBatch(escribir_bronze)
         .option("checkpointLocation", checkpoint)
         .trigger(availableNow=True)
         .queryName("bronze_usage_events").start())
    q.awaitTermination()
    return q

q = correr_stream(f"{CHK}/bronze_usage_events")
progreso = pd.DataFrame([{"batch": p["batchId"], "filas_entrada": p["numInputRows"],
                          "watermark": p.get("eventTime", {}).get("watermark")} for p in q.recentProgress])
display(progreso)

2026-11-18 19:17:49,659 | INFO | [stream] micro-lote 0: 3600 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 0: 3600 filas nuevas en Bronze
2026-11-18 19:18:15,859 | INFO | [stream] micro-lote 1: 3600 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 1: 3600 filas nuevas en Bronze
2026-11-18 19:18:43,024 | INFO | [stream] micro-lote 2: 3600 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 2: 3600 filas nuevas en Bronze
2026-11-18 19:19:07,477 | INFO | [stream] micro-lote 3: 3600 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 3: 3600 filas nuevas en Bronze
2026-11-18 19:19:33,107 | INFO | [stream] micro-lote 4: 3600 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 4: 3600 filas nuevas en Bronze
2026-11-18 19:19:55,697 | INFO | [stream] micro-lote 5: 3600 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 5: 3600 filas nuevas en Bronze
2026-11-18 19:20:20,473 | INFO | [stream] micro-lote 6: 3600 filas nuevas en Bronze
INFO:pipeline:[stream]

,batch,filas_entrada,watermark
0,0,3600,1970-01-01T00:00:00.000Z
1,1,3600,2025-07-02T22:56:00.000Z
2,2,3600,2025-07-02T22:56:00.000Z
3,3,3600,2025-07-02T22:56:00.000Z
4,4,3600,2025-07-02T22:56:00.000Z
5,5,3600,2025-07-02T22:56:00.000Z
6,6,3600,2025-07-02T22:56:00.000Z
7,7,3600,2025-07-02T22:56:00.000Z
8,8,3600,2025-07-02T22:56:00.000Z
9,9,3600,2025-07-02T22:56:00.000Z


In [8]:
# Conciliación Landing vs Bronze (evidencia de late data y deduplicación)
landing_ids = spark.read.schema(EVENT_SCHEMA).json(EVENTS_PATH).select("event_id")
bronze_ev   = spark.read.parquet(BRONZE_EVENTS)
conc = {"landing": landing_ids.count(),
        "landing_event_id_distintos": landing_ids.distinct().count(),
        "bronze": bronze_ev.count(),
        "bronze_event_id_distintos": bronze_ev.select("event_id").distinct().count()}
conc["descartados_tardios"] = conc["landing_event_id_distintos"] - conc["bronze_event_id_distintos"]
RESUMEN["stream_bronze"] = conc
print(json.dumps(conc, indent=2))
bronze_ev.groupBy("schema_version").count().show()

{
  "landing": 43200,
  "landing_event_id_distintos": 43200,
  "bronze": 43200,
  "bronze_event_id_distintos": 43200,
  "descartados_tardios": 0
}
+--------------+-----+
|schema_version|count|
+--------------+-----+
|             1|10800|
|             2|32400|
+--------------+-----+



## 3. Silver: calidad, quarantine, joins y features

### 3.1 Maestros conformados

- **customers_orgs:** se toma la foto más reciente, se normalizan región y plan, y `nps_score` fuera de -100..100 se anula y se marca (D-10). Esperamos **1** inválido (el 101).
- **billing_monthly:** tipo de cambio = 1 en facturas USD (D-07), `credits` nulo = 0 (D-08) y `revenue_usd` neto (D-13).

**Resultado esperado:** (según el perfilado de la fase 1, notebooks/01_exploracion_landing.ipynb): 1 NPS inválido (el valor 101) y 160 facturas USD corregidas.


In [9]:
cust_b = spark.read.parquet(f"{BRONZE}/customers_orgs")
ultima = cust_b.agg(F.max("ingest_date")).first()[0]
cust_s = (cust_b.filter(F.col("ingest_date") == ultima)
          .withColumn("hq_region", F.lower(F.trim("hq_region")))
          .withColumn("plan_tier", F.lower(F.trim("plan_tier")))
          .withColumn("nps_valido", F.col("nps_score").between(-100, 100))
          .withColumn("nps_score", F.when(F.col("nps_valido"), F.col("nps_score")))
          .drop("source_file", "run_id"))
cust_s.write.mode("overwrite").parquet(f"{SILVER}/customers_orgs")
nps_invalidos = cust_s.filter(F.col("nps_valido") == False).count()

bill_s = (spark.read.parquet(f"{BRONZE}/billing_monthly")
          .withColumn("fx_corregido", (F.col("currency") == "USD") & (F.col("exchange_rate_to_usd") != 1))
          .withColumn("exchange_rate_to_usd",
                      F.when(F.col("currency") == "USD", F.lit(1.0)).otherwise(F.col("exchange_rate_to_usd")))
          .withColumn("credits", F.coalesce("credits", F.lit(0.0)))
          .withColumn("revenue_usd", F.round((F.col("subtotal") - F.col("credits")) * F.col("exchange_rate_to_usd"), 2))
          .withColumn("taxes_usd", F.round(F.col("taxes") * F.col("exchange_rate_to_usd"), 2)))
bill_s.write.mode("overwrite").partitionBy("month").parquet(f"{SILVER}/billing_monthly")
fx_corregidas = bill_s.filter("fx_corregido").count()

RESUMEN["silver_maestros"] = {"customers": cust_s.count(), "nps_invalidos": nps_invalidos,
                              "facturas_fx_corregido": fx_corregidas}
print(RESUMEN["silver_maestros"])

{'customers': 80, 'nps_invalidos': 1, 'facturas_fx_corregido': 160}


### 3.2 Eventos: reglas de calidad y quarantine

Primero se normalizan textos (minúsculas y sin espacios). Después, si un evento trae `value` pero no `unit`, se intenta **imputar** la unidad a partir de la unidad más frecuente de esa misma métrica (por ejemplo, `storage_gb_hours` → `gb_hours`). Solo si no se puede imputar, el evento falla la regla.

| Regla | Qué detecta |
|---|---|
| `event_id_nulo` | Evento sin identificador |
| `registro_corrupto` | Línea JSON malformada |
| `timestamp_invalido` | Fecha no convertible |
| `value_no_numerico` | `value` presente pero no convertible a número |
| `costo_negativo` | `cost_usd_increment < -0.01` (además, flag de anomalía) |
| `unit_nula_con_value` | `value` presente y sin unidad, ni siquiera imputable |

Un evento que falla **una o más** reglas va a quarantine con todas en `dq_reason`.
**Resultado esperado:** el cruce métrica × unidad, el conteo por regla y las muestras de quarantine.

In [10]:
evn = (spark.read.parquet(BRONZE_EVENTS)
       .withColumn("region", F.lower(F.trim("region")))
       .withColumn("service", F.lower(F.trim("service")))
       .withColumn("metric", F.lower(F.trim("metric")))
       .withColumn("unit", F.lower(F.trim("unit"))))

print("Métrica × unidad (antes de imputar):")
evn.groupBy("metric", "unit").count().orderBy("metric", F.desc("count")).show(50, truncate=False)

w_metric = Window.partitionBy("metric").orderBy(F.desc("count"))
unit_map = (evn.filter("unit IS NOT NULL").groupBy("metric", "unit").count()
            .withColumn("rk", F.row_number().over(w_metric)).filter("rk = 1")
            .select("metric", F.col("unit").alias("unit_inferida")))
evn = (evn.join(F.broadcast(unit_map), "metric", "left")
          .withColumn("unit_imputada", F.col("unit").isNull() & F.col("value_num").isNotNull()
                      & F.col("unit_inferida").isNotNull())
          .withColumn("unit", F.coalesce("unit", "unit_inferida")).drop("unit_inferida"))

REGLAS = {
    "event_id_nulo":       F.col("event_id").isNull(),
    "registro_corrupto":   F.col("_corrupt_record").isNotNull(),
    "timestamp_invalido":  F.col("ts").isNull(),
    "value_no_numerico":   F.col("value").isNotNull() & F.col("value_num").isNull(),
    "costo_negativo":      F.col("cost_usd_increment") < -0.01,
    "unit_nula_con_value": F.col("value_num").isNotNull() & F.col("unit").isNull(),
}
evq = (evn.withColumn("dq_reason", F.concat_ws(";", *[F.when(c, F.lit(n)) for n, c in REGLAS.items()]))
          .withColumn("is_cost_anomaly", F.coalesce(F.col("cost_usd_increment") < -0.01, F.lit(False))))
evq.persist()

conteo_reglas = evq.agg(*[F.sum(F.coalesce(c, F.lit(False)).cast("int")).alias(n) for n, c in REGLAS.items()],
                        F.sum(F.col("unit_imputada").cast("int")).alias("units_imputadas")).first().asDict()
print("Incumplimientos por regla:", conteo_reglas)

quar = evq.filter(F.col("dq_reason") != "").withColumn("quarantine_ts", F.current_timestamp())
quar.write.mode("overwrite").partitionBy("event_date").parquet(f"{QUAR}/usage_events")
muestra_q = quar.select("event_id", "org_id", "service", "metric", "value", "unit",
                        "cost_usd_increment", "dq_reason", "source_file").limit(20).toPandas()
muestra_q.to_csv(f"{EVID}/quarantine_muestra.csv", index=False)
display(muestra_q)

Métrica × unidad (antes de imputar):
+----------------+--------+-----+
|metric          |unit    |count|
+----------------+--------+-----+
|cpu_hours       |hours   |10187|
|cpu_hours       |NULL    |505  |
|requests        |count   |18596|
|requests        |NULL    |916  |
|storage_gb_hours|gb_hours|12342|
|storage_gb_hours|NULL    |654  |
+----------------+--------+-----+

Incumplimientos por regla: {'event_id_nulo': 0, 'registro_corrupto': 0, 'timestamp_invalido': 0, 'value_no_numerico': 0, 'costo_negativo': 211, 'unit_nula_con_value': 0, 'units_imputadas': 2038}


,event_id,org_id,service,metric,value,unit,cost_usd_increment,dq_reason,source_file
0,evt_kp4vwxvp8qsw,org_chj755nf,compute,cpu_hours,3.5826,hours,-0.2877,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
1,evt_potlq161uz6o,org_fel6246h,storage,storage_gb_hours,3.5987,gb_hours,-0.0806,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
2,evt_mv9v07013r90,org_n9j2qp89,compute,requests,114.0,count,-8.1644,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
3,evt_2ie296t8aavp,org_pseimvih,storage,cpu_hours,2.6807,hours,-0.0475,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
4,evt_866mrbi9ut61,org_ofqewaou,analytics,requests,97.0,count,-7.9174,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
5,evt_kp8wwybobkgj,org_xaji0y6d,compute,cpu_hours,4.3027,hours,-0.3750,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
6,evt_oeqye72g528g,org_pbhsahxt,database,requests,131.0,count,-7.6861,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
7,evt_kwcwbndmwq2t,org_w3zp08j3,database,storage_gb_hours,20.1209,gb_hours,-0.7245,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
8,evt_wfjoor78phlh,org_5iqvnb4g,networking,storage_gb_hours,6.1126,gb_hours,-0.0636,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...
9,evt_sn88eu8yl8b8,org_ly8ozcyw,analytics,storage_gb_hours,9.8294,gb_hours,-0.7792,costo_negativo,file:///content/drive/MyDrive/Parcial%20Mineri...


In [12]:
# Eventos válidos -> Silver, enriquecidos con el maestro de clientes (join)
validos = evq.filter(F.col("dq_reason") == "")
ev_s = (validos.join(F.broadcast(cust_s.select("org_id", "org_name", "industry", "plan_tier", "hq_region")),
                     "org_id", "left")
        .drop("_corrupt_record", "dq_reason", "value", "timestamp"))
ev_s.repartition("event_date").write.mode("overwrite").partitionBy("event_date").parquet(f"{SILVER}/usage_events")

ev_s = spark.read.parquet(f"{SILVER}/usage_events")
RESUMEN["silver_eventos"] = {"bronze": evq.count(), "silver": ev_s.count(), "quarantine": quar.count(),
                             "orgs_sin_maestro": ev_s.filter("org_name IS NULL").count(),
                             "reglas": conteo_reglas}
print(json.dumps(RESUMEN["silver_eventos"], indent=2))

{
  "bronze": 43200,
  "silver": 42989,
  "quarantine": 211,
  "orgs_sin_maestro": 0,
  "reglas": {
    "event_id_nulo": 0,
    "registro_corrupto": 0,
    "timestamp_invalido": 0,
    "value_no_numerico": 0,
    "costo_negativo": 211,
    "unit_nula_con_value": 0,
    "units_imputadas": 2038
  }
}


### 3.3 Features diarias

Cada evento trae **una** métrica en `metric` y su número en `value_num`. Las features se arman filtrando por métrica y sumando por organización, servicio y día:

`daily_cost_usd`, `requests`, `cpu_hours`, `storage_gb_hours`, `genai_tokens` (solo v2) y `carbon_kg` (solo v2).

**Resultado esperado:** la lista de métricas existentes.

In [13]:
metricas = [r.metric for r in ev_s.select("metric").distinct().collect()]
print("Métricas en los datos:", metricas)

def suma_metrica(nombre):
    return F.sum(F.when(F.col("metric") == nombre, F.col("value_num")))

feat = (ev_s.groupBy("org_id", "event_date", "service")
        .agg(F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
             F.round(suma_metrica("requests"), 2).alias("requests"),
             F.round(suma_metrica("cpu_hours"), 4).alias("cpu_hours"),
             F.round(suma_metrica("storage_gb_hours"), 4).alias("storage_gb_hours"),
             F.sum("genai_tokens").alias("genai_tokens"),
             F.round(F.sum("carbon_kg"), 6).alias("carbon_kg"),
             F.count("*").alias("events")))
feat.write.mode("overwrite").partitionBy("event_date").parquet(f"{SILVER}/usage_daily_features")
feat = spark.read.parquet(f"{SILVER}/usage_daily_features")
feat.orderBy(F.desc("daily_cost_usd")).show(5)

Métricas en los datos: ['cpu_hours', 'storage_gb_hours', 'requests']
+------------+-------+--------------+--------+---------+----------------+------------+---------+------+----------+
|      org_id|service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|events|event_date|
+------------+-------+--------------+--------+---------+----------------+------------+---------+------+----------+
|org_rixa11dp|  genai|      340.5259|   390.0|   0.0473|          8.3086|        5646| 0.079671|     6|2025-08-24|
|org_c11ertj5|  genai|      334.6767|   237.0|     NULL|         12.9349|        NULL|     NULL|     3|2025-07-11|
|org_tvhhpbmy|  genai|      313.4168|   210.0|   1.3851|            NULL|        NULL|     NULL|     3|2025-07-04|
|org_x7eedtjv|compute|       311.255|   249.0|   1.8107|            NULL|        NULL|     NULL|     3|2025-07-09|
|org_5935a0l7|compute|      271.6404|   267.0|   1.1745|          7.2181|        NULL|     NULL|     6|2025-07-09|
+----------

## 4. Gold: `org_daily_usage_by_service`

Grano: **una fila por organización, día y servicio**. Se agregan el nombre y el plan de la organización para que el consumo no necesite joins.

`repartition("usage_date")` agrupa los datos de cada día en un solo archivo: 60 particiones con un archivo cada una, sin el problema de archivos chicos.

**Resultado esperado:** la conciliación. El costo total de Gold debe ser igual al de Silver.

In [14]:
gold = (feat.withColumnRenamed("event_date", "usage_date")
        .join(F.broadcast(cust_s.select("org_id", "org_name", "plan_tier")), "org_id", "left"))
GOLD_MART = f"{GOLD}/org_daily_usage_by_service"
gold.repartition("usage_date").write.mode("overwrite").partitionBy("usage_date").parquet(GOLD_MART)
gold = spark.read.parquet(GOLD_MART)

costo_silver = ev_s.agg(F.sum("cost_usd_increment")).first()[0]
costo_gold   = gold.agg(F.sum("daily_cost_usd")).first()[0]
RESUMEN["gold"] = {"filas": gold.count(), "costo_silver": round(costo_silver, 2),
                   "costo_gold": round(costo_gold, 2), "concilia": abs(costo_silver - costo_gold) < 1}
print(RESUMEN["gold"])
gold.orderBy("org_id", "usage_date", "service").show(5)

{'filas': 11050, 'costo_silver': 148351.58, 'costo_gold': 148351.58, 'concilia': True}
+------------+---------+--------------+--------+---------+----------------+------------+---------+------+-------------+---------+----------+
|      org_id|  service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|events|     org_name|plan_tier|usage_date|
+------------+---------+--------------+--------+---------+----------------+------------+---------+------+-------------+---------+----------+
|org_0lvsnujz|analytics|        8.8739|   116.0|   4.6168|         12.7808|        NULL|     NULL|     3|Nova Cloud 45| standard|2025-07-03|
|org_0lvsnujz| database|        0.5723|    NULL|   1.7515|         14.7891|        NULL|     NULL|     3|Nova Cloud 45| standard|2025-07-03|
|org_0lvsnujz|  storage|        5.5542|   255.0|     NULL|          4.6668|        NULL|     NULL|     3|Nova Cloud 45| standard|2025-07-03|
|org_0lvsnujz|analytics|        3.2915|    NULL|   2.7659|         

## 5. Analítica: detección de anomalías de costo (MAD)

Resuelve la decisión abierta D-12. La exploración mostró costos muy asimétricos (máximo 19 veces el p99). El z-score clásico usa media y desvío, que los propios picos inflan. Por eso usamos el **z-score robusto**, basado en la mediana y en el MAD (desvío absoluto mediano), calculado por organización y servicio:

`z_robusto = 0,6745 × (costo − mediana) / MAD`

Se marca anomalía cuando `z_robusto > 3,5`, el umbral habitual de Iglewicz y Hoaglin. Solo se marcan picos hacia arriba, que son los que preocupan a FinOps; los costos negativos ya quedaron marcados en quarantine.

**Resultado esperado:** cuántos días se marcan y si el top coincide con los picos que vimos en la exploración.

In [15]:
med = gold.groupBy("org_id", "service").agg(F.percentile_approx("daily_cost_usd", 0.5).alias("mediana"))
g2  = gold.join(med, ["org_id", "service"]).withColumn("desvio_abs", F.abs(F.col("daily_cost_usd") - F.col("mediana")))
mad = g2.groupBy("org_id", "service").agg(F.percentile_approx("desvio_abs", 0.5).alias("mad"))
anom = (g2.join(mad, ["org_id", "service"])
        .withColumn("z_robusto", F.when(F.col("mad") > 0,
                    F.round(0.6745 * (F.col("daily_cost_usd") - F.col("mediana")) / F.col("mad"), 2)))
        .withColumn("is_anomaly", F.coalesce(F.col("z_robusto") > ANOMALY_Z, F.lit(False)))
        .select("org_id", F.col("usage_date").alias("date"), "service", "daily_cost_usd",
                "mediana", "mad", "z_robusto", "is_anomaly"))
anom.write.mode("overwrite").partitionBy("date").parquet(f"{GOLD}/cost_anomaly_mart")
anom = spark.read.parquet(f"{GOLD}/cost_anomaly_mart")

n_anom = anom.filter("is_anomaly").count()
RESUMEN["anomalias"] = {"metodo": "z-score robusto (MAD)", "umbral": ANOMALY_Z,
                        "dias_evaluados": anom.count(), "anomalias": n_anom}
print(RESUMEN["anomalias"])
anom.filter("is_anomaly").orderBy(F.desc("z_robusto")).show(10)

{'metodo': 'z-score robusto (MAD)', 'umbral': 3.5, 'dias_evaluados': 11050, 'anomalias': 139}
+------------+----------+--------------+-------+------------------+---------+----------+----------+
|      org_id|   service|daily_cost_usd|mediana|               mad|z_robusto|is_anomaly|      date|
+------------+----------+--------------+-------+------------------+---------+----------+----------+
|org_okep7y6w|   compute|       202.646|11.1426|2.7475000000000005|    47.01|      true|2025-07-03|
|org_x7eedtjv|   compute|       311.255|12.9288| 4.482200000000001|    44.89|      true|2025-07-09|
|org_okep7y6w|   compute|      191.8968|11.1426|2.7475000000000005|    44.37|      true|2025-08-07|
|org_w3zp08j3|networking|       27.4203| 1.4849|0.4557000000000002|    38.39|      true|2025-07-05|
|org_w3zp08j3|networking|       26.4157| 1.4849|0.4557000000000002|     36.9|      true|2025-07-16|
|org_d14ve92m| analytics|      181.7988|10.5941| 3.216799999999999|     35.9|      true|2025-07-20|
|org_p

## 6. Serving en Cassandra / AstraDB

### 6.1 Credenciales (nunca en el código)

1. En Colab, abrí el panel de la **llave** (🔑 *Secretos*, a la izquierda).
2. Agregá un secreto llamado `ASTRA_TOKEN` con tu token `AstraCS:...` y activá el acceso al notebook.
3. Verificá que el bundle esté en `ASTRA_SCB` (sección 0). Si el archivo tiene otro nombre, cambiá esa variable.

### 6.2 Modelo query-first

La consulta que resuelve la tabla es: *costos y requests diarios de una organización por servicio, en un rango de fechas*.

```
PRIMARY KEY ((org_id), usage_date, service)  — CLUSTERING ORDER BY (usage_date DESC, service ASC)
```

- **Partición `org_id`:** la consulta siempre filtra por una organización, así que se lee una sola partición.
- **Clustering `usage_date DESC`:** permite el rango de fechas y devuelve primero lo más reciente.
- **`service`:** hace única la fila. Cargar dos veces la misma fila la **sobrescribe** (upsert), lo que vuelve idempotente la carga.
- **Tamaño de partición:** 6 servicios × 365 días ≈ 2.200 filas por año por organización, muy por debajo del umbral de alerta.

In [16]:
from google.colab import userdata
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.concurrent import execute_concurrent_with_args

ASTRA_TOKEN = userdata.get("ASTRA_TOKEN")
assert os.path.isfile(ASTRA_SCB), f"No encuentro el bundle: {ASTRA_SCB}"
cluster = Cluster(cloud={"secure_connect_bundle": ASTRA_SCB},
                  auth_provider=PlainTextAuthProvider("token", ASTRA_TOKEN))
session = cluster.connect(ASTRA_KEYSPACE)
session.default_timeout = 60
print("Conectado. Versión:", session.execute("SELECT release_version FROM system.local").one().release_version)

DDL = """
CREATE TABLE IF NOT EXISTS org_daily_usage_by_service (
    org_id text, usage_date date, service text,
    org_name text, plan_tier text,
    daily_cost_usd double, requests double, cpu_hours double, storage_gb_hours double,
    genai_tokens bigint, carbon_kg double, events int, run_id text,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date DESC, service ASC)"""
session.execute(DDL)
print("Tabla lista.")

Conectado. Versión: 4.0.11.0-6819aa9c7f6c
Tabla lista.


In [17]:
INSERT = session.prepare("""
INSERT INTO org_daily_usage_by_service (org_id, usage_date, service, org_name, plan_tier,
    daily_cost_usd, requests, cpu_hours, storage_gb_hours, genai_tokens, carbon_kg, events, run_id)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""")

def cargar_cassandra():
    filas = [(r.org_id, r.usage_date, r.service, r.org_name, r.plan_tier, r.daily_cost_usd,
              r.requests, r.cpu_hours, r.storage_gb_hours,
              int(r.genai_tokens) if r.genai_tokens is not None else None,
              r.carbon_kg, int(r.events), RUN_ID)
             for r in spark.read.parquet(GOLD_MART).toLocalIterator()]
    res = execute_concurrent_with_args(session, INSERT, filas, concurrency=50, raise_on_first_error=False)
    fallos = sum(1 for ok, _ in res if not ok)
    log.info(f"[serving] filas enviadas={len(filas)} fallos={fallos}")
    return len(filas), fallos

ORG_IDS = [r.org_id for r in gold.select("org_id").distinct().collect()]
def contar_cassandra():
    q = session.prepare("SELECT COUNT(*) FROM org_daily_usage_by_service WHERE org_id = ?")
    return sum(session.execute(q, (o,)).one()[0] for o in ORG_IDS)

enviadas, fallos = cargar_cassandra()
RESUMEN["serving"] = {"filas_gold": gold.count(), "enviadas": enviadas, "fallos": fallos,
                      "filas_cassandra": contar_cassandra()}
print(RESUMEN["serving"])

2026-11-18 20:01:10,845 | INFO | [serving] filas enviadas=11050 fallos=0
INFO:pipeline:[serving] filas enviadas=11050 fallos=0


{'filas_gold': 11050, 'enviadas': 11050, 'fallos': 0, 'filas_cassandra': 11050}


### 6.3 Dos consultas mínimas

- **Q1:** costos y requests diarios de una organización por servicio en los últimos 14 días del dataset.
- **Q2:** detalle por servicio de esa organización en su día de mayor costo.

Ambas leen **una sola partición** (`org_id`), sin `ALLOW FILTERING`. Además, con el resultado de Q1 se muestra el top de servicios por costo en 14 días.

**Evidencia:** los resultados se guardan en `_evidence/`.

In [18]:
ORG = gold.groupBy("org_id").agg(F.sum("daily_cost_usd").alias("c")).orderBy(F.desc("c")).first().org_id
fecha_max = gold.agg(F.max("usage_date")).first()[0]
desde = fecha_max - dt.timedelta(days=13)

Q1 = ("SELECT usage_date, service, daily_cost_usd, requests FROM org_daily_usage_by_service "
      "WHERE org_id = %s AND usage_date >= %s AND usage_date <= %s")
q1 = pd.DataFrame(list(session.execute(Q1, (ORG, desde, fecha_max))))
print(f"Q1 · org={ORG} · {desde} a {fecha_max} · filas={len(q1)}")
display(q1.head(15))

dia_pico = q1.groupby("usage_date")["daily_cost_usd"].sum().idxmax()
Q2 = ("SELECT service, daily_cost_usd, requests, events FROM org_daily_usage_by_service "
      "WHERE org_id = %s AND usage_date = %s")
q2 = pd.DataFrame(list(session.execute(Q2, (ORG, dia_pico))))
print(f"Q2 · org={ORG} · día={dia_pico}")
display(q2)

top = q1.groupby("service")["daily_cost_usd"].sum().sort_values(ascending=False).round(2)
print("Top servicios por costo en 14 días (derivado de Q1):"); display(top)

q1.to_csv(f"{EVID}/q1_resultado.csv", index=False); q2.to_csv(f"{EVID}/q2_resultado.csv", index=False)
print("\nPara la CQL Console de Astra:")
print(f"SELECT usage_date, service, daily_cost_usd, requests FROM {ASTRA_KEYSPACE}.org_daily_usage_by_service "
      f"WHERE org_id = '{ORG}' AND usage_date >= '{desde}' AND usage_date <= '{fecha_max}';")

Q1 · org=org_53lc58dr · 2025-08-18 a 2025-08-31 · filas=68


,usage_date,service,daily_cost_usd,requests
0,2025-08-31,analytics,47.3679,604.0
1,2025-08-31,compute,1.0125,NaN
2,2025-08-31,database,1.7011,NaN
3,2025-08-31,genai,15.1432,112.0
4,2025-08-31,storage,2.6436,125.0
5,2025-08-30,compute,11.3755,112.0
6,2025-08-30,database,19.2887,365.0
7,2025-08-30,genai,33.0729,243.0
8,2025-08-30,networking,3.4050,321.0
9,2025-08-30,storage,2.6326,114.0


Q2 · org=org_53lc58dr · día=2025-08-23


,service,daily_cost_usd,requests,events
0,analytics,32.8759,493.0,6
1,compute,11.5343,120.0,3
2,database,19.4919,342.0,6
3,genai,55.4541,364.0,6
4,storage,5.2288,228.0,3


Top servicios por costo en 14 días (derivado de Q1):


,daily_cost_usd
service,
analytics,282.07
genai,271.27
compute,237.48
database,182.78
storage,36.27
networking,15.43



Para la CQL Console de Astra:
SELECT usage_date, service, daily_cost_usd, requests FROM cloud_analytics.org_daily_usage_by_service WHERE org_id = 'org_53lc58dr' AND usage_date >= '2025-08-18' AND usage_date <= '2025-08-31';


## 7. Idempotencia: re-ejecutar sin duplicar

Se toman conteos de cada capa, se **re-ejecuta todo** y se vuelve a contar:

1. Batch: misma fecha de carga → `overwrite` dinámico reemplaza la misma partición.
2. Streaming con el **mismo checkpoint** → Spark sabe qué archivos ya procesó y no lee nada.
3. Streaming con un checkpoint **nuevo** (reproceso desde cero) → el anti-join por `event_id` evita duplicados.
4. Recarga en Cassandra → la clave primaria convierte los inserts en upserts.

**Resultado esperado:** la columna `igual`. Todo tiene que dar `True`.

In [19]:
def conteos():
    return {"bronze_customers": spark.read.parquet(f"{BRONZE}/customers_orgs").count(),
            "bronze_users": spark.read.parquet(f"{BRONZE}/users").count(),
            "bronze_billing": spark.read.parquet(f"{BRONZE}/billing_monthly").count(),
            "bronze_eventos": spark.read.parquet(BRONZE_EVENTS).count(),
            "gold_mart": spark.read.parquet(GOLD_MART).count(),
            "cassandra": contar_cassandra()}

antes = conteos()
for n in MAESTROS: ingestar_maestro(n)
correr_stream(f"{CHK}/bronze_usage_events")             # mismo checkpoint
correr_stream(f"{CHK}/bronze_usage_events_reproceso")   # checkpoint nuevo: reproceso total
cargar_cassandra()
despues = conteos()

idem = pd.DataFrame({"antes": antes, "despues": despues})
idem["igual"] = idem["antes"] == idem["despues"]
RESUMEN["idempotencia"] = idem.reset_index().rename(columns={"index": "capa"}).to_dict("records")
idem.to_csv(f"{EVID}/idempotencia.csv")
display(idem)

2026-11-18 20:06:47,662 | INFO | [batch] customers_orgs: leidas=80 escritas=80
INFO:pipeline:[batch] customers_orgs: leidas=80 escritas=80
2026-11-18 20:06:48,987 | INFO | [batch] users: leidas=800 escritas=800
INFO:pipeline:[batch] users: leidas=800 escritas=800
2026-11-18 20:06:50,210 | INFO | [batch] billing_monthly: leidas=240 escritas=240
INFO:pipeline:[batch] billing_monthly: leidas=240 escritas=240
2026-11-18 20:07:42,627 | INFO | [stream] micro-lote 0: 0 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 0: 0 filas nuevas en Bronze
2026-11-18 20:08:28,870 | INFO | [stream] micro-lote 1: 0 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 1: 0 filas nuevas en Bronze
2026-11-18 20:08:59,510 | INFO | [stream] micro-lote 2: 0 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 2: 0 filas nuevas en Bronze
2026-11-18 20:09:30,312 | INFO | [stream] micro-lote 3: 0 filas nuevas en Bronze
INFO:pipeline:[stream] micro-lote 3: 0 filas nuevas en Bronze
2026-11-18 20:10:00

,antes,despues,igual
bronze_customers,80,80,True
bronze_users,800,800,True
bronze_billing,240,240,True
bronze_eventos,43200,43200,True
gold_mart,11050,11050,True
cassandra,11050,11050,True


## 8. Evidencias, tamaños y persistencia

Se guardan en `_evidence/`: el resumen de la ejecución (`run_summary.json`), los tamaños y rutas de cada tabla, el log, las muestras de quarantine y los resultados de las consultas. Después, todo el lake se copia a Drive.


In [20]:
tablas = [f"{BRONZE}/{t}" for t in ("customers_orgs", "users", "billing_monthly", "usage_events")] + \
         [f"{SILVER}/{t}" for t in ("customers_orgs", "billing_monthly", "usage_events", "usage_daily_features")] + \
         [f"{GOLD}/org_daily_usage_by_service", f"{GOLD}/cost_anomaly_mart", f"{QUAR}/usage_events"]
tam = pd.DataFrame([tamano_tabla(t) for t in tablas])
tam.to_csv(f"{EVID}/tamanos_rutas.csv", index=False)
display(tam)

RESUMEN["tamanos"] = tam.to_dict("records")
with open(f"{EVID}/run_summary.json", "w") as f:
    json.dump(RESUMEN, f, indent=2, default=str)

shutil.copytree(LAKE, DRIVE_OUT, dirs_exist_ok=True)
log.info(f"Lake copiado a {DRIVE_OUT}")
print("Listo. Evidencias en:", f"{DRIVE_OUT}/_evidence")

,tabla,archivos,particiones,MB
0,bronze/customers_orgs,1,1,0.008
1,bronze/users,1,1,0.025
2,bronze/billing_monthly,3,3,0.024
3,bronze/usage_events,5757,60,41.506
4,silver/customers_orgs,1,1,0.007
5,silver/billing_monthly,3,3,0.028
6,silver/usage_events,60,60,2.667
7,silver/usage_daily_features,60,60,0.560
8,gold/org_daily_usage_by_service,60,60,0.656
9,gold/cost_anomaly_mart,60,60,0.519


2026-11-18 20:18:29,308 | INFO | Lake copiado a /content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto/datalake_output
INFO:pipeline:Lake copiado a /content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto/datalake_output


Listo. Evidencias en: /content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto/datalake_output/_evidence


In [21]:
from google.colab import files
print(os.listdir(EVID))                                   # lista de evidencias generadas
shutil.make_archive("/content/evidence", "zip", EVID)     # las comprime en un .zip
files.download("/content/evidence.zip")                   # las descarga a tu computadora

['run_summary.json', 'run_20261118T190912Z.log', 'quarantine_muestra.csv', 'q1_resultado.csv', 'tamanos_rutas.csv', 'idempotencia.csv', 'simulacion_watermark.csv', 'q2_resultado.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Limpieza y reinicio

- **Cerrar conexiones** al terminar: celda siguiente.
- **Reiniciar desde cero:** descomentá las líneas marcadas. Borra el lake local y los checkpoints, y vacía la tabla de Cassandra. El Landing **nunca** se toca.

In [22]:
for s in spark.streams.active:
    s.stop()
cluster.shutdown()
print("Conexiones cerradas.")

# --- Reinicio completo (descomentar solo si se quiere empezar de cero) ---
# shutil.rmtree(LAKE, ignore_errors=True)
# session = Cluster(cloud={"secure_connect_bundle": ASTRA_SCB},
#                   auth_provider=PlainTextAuthProvider("token", ASTRA_TOKEN)).connect(ASTRA_KEYSPACE)
# session.execute("TRUNCATE org_daily_usage_by_service")

Conexiones cerradas.
